# Saddles, SFN, and training curves

Start with the geometry of $f(x,y)=5x^2-y^2$, then inspect runs saved by
`src/main.py`. The toy example uses the actual `SFN` optimizer in this project.
It illustrates the local direction of motion, rather than reproducing a paper figure.

Use the Python kernel from the project's `.venv`. No cell downloads datasets or
launches a training experiment. See the README for training commands.

In [ ]:
import csv
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "pyproject.toml").is_file() and (directory / "src/main.py").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from descent_algorithms import SFN, SGD  # noqa: E402 - locate the project first

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
print(f"Project: {PROJECT_ROOT.name} | PyTorch: {torch.__version__}")

## 1. Follow a quadratic saddle

Here $H=\operatorname{diag}(10,-2)$. Newton divides by signed eigenvalues and
moves directly to the origin. SFN divides by their magnitudes and moves away in
the negative-curvature direction. Gradient descent also moves away, with its step
size directly dependent on curvature.

The methods use explicitly chosen illustration step sizes below. They are not a
comparison of tuned convergence speed. The starting point has nonzero gradient;
an exact stationary point would give a zero SFN step.

In [ ]:
def objective(point):
    return 5 * point[0] ** 2 - point[1] ** 2


def trajectory(method, steps=6):
    point = torch.nn.Parameter(torch.tensor([0.9, 0.08], dtype=torch.float64))
    if method == "Gradient descent":
        optimizer = SGD([point], lr=0.08)
    elif method == "SFN":
        optimizer = SFN([point], lr=0.5, damping=0.1, batch_size=1)
    history = [point.detach().clone()]
    for _ in range(steps):
        if method == "Newton":
            gradient = torch.autograd.grad(objective(point), point)[0]
            hessian = torch.autograd.functional.hessian(objective, point)
            with torch.no_grad():
                point.sub_(torch.linalg.solve(hessian, gradient))
        elif method == "SFN":
            optimizer.step(lambda: objective(point))
        else:
            optimizer.zero_grad(set_to_none=True)
            objective(point).backward()
            optimizer.step()
        history.append(point.detach().clone())
    return torch.stack(history)


coordinates = torch.linspace(-1.1, 1.1, 160, dtype=torch.float64)
x, y = torch.meshgrid(coordinates, coordinates, indexing="xy")
fig, ax = plt.subplots(figsize=(7, 5))
ax.contour(x, y, 5 * x**2 - y**2, levels=25, cmap="Greys", alpha=0.45)
for method in ["Gradient descent", "Newton", "SFN"]:
    path = trajectory(method)
    ax.plot(path[:, 0], path[:, 1], "o-", markersize=4, label=method)
ax.scatter([0], [0], marker="x", s=90, color="black", label="Saddle")
ax.set(xlabel="x: positive curvature", ylabel="y: negative curvature", title="Motion near a saddle")
ax.set_aspect("equal")
ax.legend()
fig.tight_layout()
plt.show()

## 2. Find saved experiments

Run the README commands first, then rerun these cells. Set `DATASET` to `mnist`
or `cifar10`. You can replace `RUN_DIRS` with specific run folders to control
which experiments appear. This notebook reads metrics; it never trains a model.

Use matching model sizes, data subsets, seeds, and preprocessing for comparisons.
The code reports differing configurations so that unrelated runs are visible.

In [ ]:
DATASET = "mnist"
RUN_DIRS = sorted((PROJECT_ROOT / "results").glob(f"{DATASET}_*"))
runs = []
for directory in RUN_DIRS:
    if not (directory / "metrics.csv").is_file():
        continue
    config = json.loads((directory / "config.json").read_text())
    with (directory / "metrics.csv").open() as stream:
        metrics = [
            {key: float(value) for key, value in row.items()} for row in csv.DictReader(stream)
        ]
    if metrics:
        runs.append((directory, config, metrics))
        print(directory.name)

if not runs:
    print("No runs yet. Run a training command in the README, then rerun this cell.")
else:
    comparable = [
        "dataset",
        "image_size",
        "hidden_units",
        "train_limit",
        "eval_limit",
        "seed",
        "dtype",
    ]
    for key in comparable:
        values = {str(config[key]) for _, config, _ in runs}
        if len(values) > 1:
            print(f"Different {key} values: {sorted(values)}")

## 3. Compare loss, accuracy, and training error

Choose `X_AXIS = "epoch"`, `"updates"`, or `"train_seconds"`. Epochs measure
passes over the training subset; updates measure parameter steps; training seconds
include data loading and optimization and exclude evaluation. SFN steps are much
more expensive, and batch size changes how many steps fit in one epoch.

The training-error panel converts stored accuracy to percent, making it closer
to the quantity shown in the paper's small-network experiments. These initial
runs are workflow checks, not a reproduction of the paper's results.

In [ ]:
X_AXIS = "epoch"
axis_labels = {
    "epoch": "Epoch",
    "updates": "Optimizer updates",
    "train_seconds": "Training seconds",
}
if X_AXIS not in axis_labels:
    raise ValueError(f"Choose one of {list(axis_labels)}")

if runs:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for directory, config, metrics in runs:
        x_values = [row[X_AXIS] for row in metrics]
        label = f"{config['algorithm']} (seed {config['seed']}, {directory.name[-8:]})"
        axes[0].plot(x_values, [row["train_loss"] for row in metrics], marker=".", label=label)
        axes[1].plot(x_values, [100 * row["validation_accuracy"] for row in metrics], marker=".")
        axes[2].plot(x_values, [100 * (1 - row["train_accuracy"]) for row in metrics], marker=".")
    for ax, title, ylabel in zip(
        axes,
        ["Training loss", "Validation accuracy", "Training error"],
        ["Mean cross entropy", "Accuracy (%)", "Classification error (%)"],
        strict=True,
    ):
        ax.set(xlabel=axis_labels[X_AXIS], ylabel=ylabel, title=title)
        ax.grid(alpha=0.2)
    axes[1].set_ylim(0, 100)
    axes[2].set_ylim(0, 100)
    axes[0].legend(fontsize=7)
    fig.tight_layout()
    plt.show()

## Questions to explore next

- How does damping change SFN's step along a nearly flat direction?
- How much does the minibatch change the Hessian spectrum and the resulting step?
- Does an apparent plateau have both a small gradient and negative curvature?
- Do conclusions change when plotting against time instead of epochs?
- How well does a small Krylov subspace approximate the exact SFN step?

The README connects these questions to the paper's sections and appendices.